# ORB bracket grid search — why 1R and what beats it?

**Production desk (paper)** follows Michael Zheng’s *Full-Stack-Stock-Algorithm* / hackingthemarkets ORB port in `pipeline/compute/opening_range.py`:

1. **Gappers:** prior close → open ≥ **4%** (long-only in production)
2. **Opening range:** **09:30–09:45 ET** on **5-minute** bars
3. **Entry:** first 5m **close** after 09:45 **above** OR high
4. **Bracket:** stop = entry − **1×OR**, target = entry + **1×OR** → **1R reward for 1R risk** (symmetric box)
5. **Size:** `floor(equity × 5% / price)` per name, up to **8** gappers

**Why 1R?** It is the reference implementation default—not an optimality claim. A 1:1 bracket is easy to paper-log and matches the upstream repo; many ORB traders use wider targets (2R–3R) or OR-low stops instead.

This notebook **grid-searches** gap filter, OR length, and **stop/target R-multiples** on the same rules, then plots **equity curves** and metrics.

**Data limit:** Yahoo `5m` bars are only available for roughly the **last 60 days**. This is *not* the multi-year IS/OOS split used by Donchian/LOOP notebooks—we split the available window **55% IS / 45% OOS** instead.

No transaction costs or slippage. Not investment advice.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.research.orb_backtest import (
    OrbParams,
    grid_search_orb,
    orb_sample_split,
    run_orb_backtest,
    trading_sessions_in_daily,
    intraday_available_sessions,
    _download_daily,
)
from pipeline.compute.opening_range import LIQUID

LOOKBACK_DAYS = 65
TICKERS = list(LIQUID)
print(f"Universe: {len(TICKERS)} liquid names (same seed list as live ORB)")

## Baseline — production 1R / 1R bracket

In [ ]:
baseline = OrbParams()  # gap 4%, 15m OR, stop 1R, target 1R
trades, daily_ret, eq = run_orb_backtest(
    baseline, tickers=TICKERS, lookback_days=LOOKBACK_DAYS, clear_cache=True
)

daily = _download_daily(TICKERS, lookback_days=LOOKBACK_DAYS)
allowed = set(intraday_available_sessions())
sessions = [d for d in trading_sessions_in_daily(daily) if d in allowed]
(is_start, is_end), (oos_start, oos_end) = orb_sample_split(sessions)
print(f"Sample: {sessions[0]} → {sessions[-1]}  |  IS {is_start}→{is_end}  OOS {oos_start}→{oos_end}")
print(f"Trades: {len(trades)}  |  Final equity: ${eq['equity'].iloc[-1]:,.0f}")
if not trades.empty:
    display(trades.groupby('exit_reason').size().to_frame('count'))
    display(trades[['session', 'symbol', 'gap_pct', 'r_multiple', 'pnl', 'exit_reason']].tail(10))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
eq_plot = eq.copy()
eq_plot.index = pd.to_datetime(eq_plot.index)
ax.plot(eq_plot.index, eq_plot['equity'], label='ORB 1R/1R (baseline)', color='#3b82f6', lw=2)
ax.axvline(pd.Timestamp(oos_start), color='#f59e0b', ls='--', alpha=0.8, label='OOS start')
ax.set_title('ORB paper equity — baseline 1R bracket')
ax.set_ylabel('Equity ($)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Grid search

Set `FULL_GRID = True` for the complete sweep (~600 combos, several minutes). Default is a **fast** grid for interactive use.

In [ ]:
FULL_GRID = False

if FULL_GRID:
    grid_df = grid_search_orb(tickers=TICKERS, lookback_days=LOOKBACK_DAYS)
else:
    fast = []
    for gap in (3.0, 4.0, 5.0, 6.0):
        for or_m in (10, 15, 20, 30):
            for target_r in (0.75, 1.0, 1.5, 2.0, 3.0):
                fast.append(OrbParams(gap_pct=gap, or_minutes=or_m, stop_r=1.0, target_r=target_r))
    grid_df = grid_search_orb(fast, tickers=TICKERS, lookback_days=LOOKBACK_DAYS)

print(f"Grid rows: {len(grid_df)}")
cols = [
    'gap_pct', 'or_minutes', 'stop_r', 'target_r', 'n_trades', 'win_rate', 'avg_r',
    'total_return_pct', 'sharpe', 'max_drawdown_pct', 'oos_return_pct', 'oos_sharpe',
]
display(grid_df[cols].head(15))

In [ ]:
best = grid_df.iloc[0]
best_params = OrbParams(
    gap_pct=float(best['gap_pct']),
    or_minutes=int(best['or_minutes']),
    stop_r=float(best['stop_r']),
    target_r=float(best['target_r']),
)
print('Best by OOS Sharpe:', best_params)

best_trades, _, best_eq = run_orb_backtest(
    best_params, tickers=TICKERS, lookback_days=LOOKBACK_DAYS, clear_cache=True
)

fig, ax = plt.subplots(figsize=(10, 4))
for label, frame, color in [
    ('Baseline 1R/1R', eq, '#64748b'),
    (f"Best grid (TP {best_params.target_r}R)", best_eq, '#22c55e'),
]:
    f = frame.copy()
    f.index = pd.to_datetime(f.index)
    ax.plot(f.index, f['equity'], label=label, color=color, lw=2)
ax.axvline(pd.Timestamp(oos_start), color='#f59e0b', ls='--', alpha=0.8, label='OOS start')
ax.set_title('ORB equity — baseline vs grid best (OOS Sharpe)')
ax.set_ylabel('Equity ($)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: target R vs gap filter (collapse OR length — mean OOS Sharpe)
pivot = grid_df.groupby(['gap_pct', 'target_r'])['oos_sharpe'].mean().unstack('target_r')
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(i) for i in pivot.index])
ax.set_xlabel('Target (R multiples of OR width)')
ax.set_ylabel('Gap filter (%)')
ax.set_title('Mean OOS Sharpe — gap × target R (avg over OR lengths)')
plt.colorbar(im, ax=ax, label='OOS Sharpe')
plt.tight_layout()
plt.show()

## Takeaways

- **1R/1R** is the **upstream default**, not a tuned optimum.
- On the **~60-day Yahoo window**, compare **OOS Sharpe** and **max drawdown** before changing production brackets.
- To apply a winner live, set env `ORB_GAP_PCT` (gap filter) and extend `opening_range.py` with `ORB_TARGET_R` / `ORB_STOP_R` if you want non-1R brackets in the desk.